<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/Phenotype_Aware_MODEL_OMI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — Phenotype-Aware Loss Weighting for OMI Detection

This experiment keeps the selected Task-1 baseline fixed and changes only the training loss weighting.

Weighting policy:
- all non-OMI: 1.0×
- STEMI-OMI: 1.0×
- NSTEMI-OMI: 1.5×
- NSTEMI-OMI with LCX or RCA involvement: 2.0×

Task 3 uses normal shuffled sampling. Hard-negative sampling is disabled so this remains a clean ablation.


In [1]:
import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# CELL 1 — IMPORTS, PATHS, DEVICE

import os
import random
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

PRETRAINED_PATH = os.path.join(
    PROJECT_DIR,
    "ECGFounder",
    "12_lead_ECGFounder.pth"
)

BASE = os.path.join(
    PROJECT_DIR,
    "results",
    "preprocessing",
    "processed"
)

TRAIN_H5 = f"{BASE}/train_fixed.h5"
VAL_H5 = f"{BASE}/validation.h5"

TRAIN_CSV = f"{BASE}/train_omi.csv"
VAL_CSV = f"{BASE}/validation_omi.csv"

SAVE_DIR = os.path.join(
    PROJECT_DIR,
    "OMI_model",
    "task3_phenotype_weighting"
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Save directory:", SAVE_DIR)


Device: cuda
Save directory: /content/drive/MyDrive/ECG_Project/OMI_model/task3_phenotype_weighting


In [3]:
# CELL 2 — CONFIG

CONFIG = {
    "baseline_mode": "last2",

    "head_lr": 1e-4,
    "backbone_lr": 1e-5,

    "weight_decay": 1e-4,

    "epochs": 30,
    "patience": 5,

    "max_grad_norm": 5.0,
    "batch_size": 16,

    "nstemi_omi_weight": 1.5,
    "hard_nstemi_vessel_weight": 2.0,

    "save_dir": SAVE_DIR
}

print(CONFIG)


{'baseline_mode': 'last2', 'head_lr': 0.0001, 'backbone_lr': 1e-05, 'weight_decay': 0.0001, 'epochs': 30, 'patience': 5, 'max_grad_norm': 5.0, 'batch_size': 16, 'nstemi_omi_weight': 1.5, 'hard_nstemi_vessel_weight': 2.0, 'save_dir': '/content/drive/MyDrive/ECG_Project/OMI_model/task3_phenotype_weighting'}


In [4]:
# CELL 3 — ECGFOUNDER ARCHITECTURE

class MyConv1dPadSame(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            groups=groups
        )

    def forward(self, x):
        in_dim = x.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        left = p // 2
        right = p - left
        x = F.pad(x, (left, right), "constant", 0)
        return self.conv(x)


class MyMaxPool1dPadSame(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.max_pool = nn.MaxPool1d(kernel_size=kernel_size)

    def forward(self, x):
        p = max(0, self.kernel_size - 1)
        left = p // 2
        right = p - left
        x = F.pad(x, (left, right), "constant", 0)
        return self.max_pool(x)


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        ratio,
        kernel_size,
        stride,
        groups,
        downsample,
        is_first_block=False,
        use_bn=True,
        use_do=True
    ):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.downsample = downsample
        self.stride = stride if downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        middle_channels = int(out_channels * ratio)

        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(in_channels, middle_channels, 1, 1, 1)

        self.bn2 = nn.BatchNorm1d(middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            middle_channels,
            middle_channels,
            kernel_size,
            self.stride,
            groups
        )

        self.bn3 = nn.BatchNorm1d(middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(middle_channels, out_channels, 1, 1, 1)

        r = 2
        self.se_fc1 = nn.Linear(out_channels, out_channels // r)
        self.se_fc2 = nn.Linear(out_channels // r, out_channels)
        self.se_activation = Swish()

        if downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):
        identity = x
        out = x

        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)

        out = self.conv1(out)

        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)

        out = self.conv2(out)

        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)

        out = self.conv3(out)

        se = out.mean(-1)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se)

        out = torch.einsum("abc,ab->abc", out, se)

        if self.downsample:
            identity = self.max_pool(identity)

        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1, -2)
            ch1 = (self.out_channels - self.in_channels) // 2
            ch2 = self.out_channels - self.in_channels - ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1, -2)

        out += identity
        return out


class BasicStage(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        ratio,
        kernel_size,
        stride,
        groups,
        i_stage,
        m_blocks,
        use_bn=True,
        use_do=True
    ):
        super().__init__()

        self.block_list = nn.ModuleList()

        for i_block in range(m_blocks):
            is_first_block = (i_stage == 0 and i_block == 0)
            downsample = (i_block == 0)
            block_stride = stride if downsample else 1
            block_in = in_channels if i_block == 0 else out_channels

            self.block_list.append(
                BasicBlock(
                    in_channels=block_in,
                    out_channels=out_channels,
                    ratio=ratio,
                    kernel_size=kernel_size,
                    stride=block_stride,
                    groups=groups,
                    downsample=downsample,
                    is_first_block=is_first_block,
                    use_bn=use_bn,
                    use_do=use_do
                )
            )

    def forward(self, x):
        for block in self.block_list:
            x = block(x)
        return x


class Net1D(nn.Module):
    def __init__(
        self,
        in_channels,
        base_filters,
        ratio,
        filter_list,
        m_blocks_list,
        kernel_size,
        stride,
        groups_width,
        n_classes,
        use_bn=True,
        use_do=True,
        return_features=False
    ):
        super().__init__()

        self.use_bn = use_bn
        self.return_features = return_features
        self.n_stages = len(filter_list)

        self.first_conv = MyConv1dPadSame(
            in_channels,
            base_filters,
            kernel_size,
            2
        )

        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        self.stage_list = nn.ModuleList()
        current_in = base_filters

        for i_stage, out_channels in enumerate(filter_list):
            self.stage_list.append(
                BasicStage(
                    in_channels=current_in,
                    out_channels=out_channels,
                    ratio=ratio,
                    kernel_size=kernel_size,
                    stride=stride,
                    groups=out_channels // groups_width,
                    i_stage=i_stage,
                    m_blocks=m_blocks_list[i_stage],
                    use_bn=use_bn,
                    use_do=use_do
                )
            )

            current_in = out_channels

        self.dense = nn.Linear(current_in, n_classes)

    def forward(self, x):
        x = self.first_conv(x)

        if self.use_bn:
            x = self.first_bn(x)

        x = self.first_activation(x)

        for stage in self.stage_list:
            x = stage(x)

        deep_features = x.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features

        return out


In [5]:
# CELL 4 — LOAD ORIGINAL PRETRAINED ECGFOUNDER + FRESH OMI HEAD

model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=150,
    use_bn=False,
    use_do=False,
    return_features=True
)

checkpoint = torch.load(
    PRETRAINED_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["state_dict"])

model.dense = nn.Linear(1024, 1)
model.return_features = False
model = model.to(DEVICE)

print("Original ECGFounder loaded.")
print("Fresh OMI head:", model.dense)


Original ECGFounder loaded.
Fresh OMI head: Linear(in_features=1024, out_features=1, bias=True)


In [6]:
# ============================================================
# CELL 5 — LOAD METADATA + BUILD PHENOTYPE WEIGHTS
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

required_cols = [
    "OMI",
    "STEMI",
    "NSTEMI",
    "PLAD",
    "MLAD",
    "DLAD",
    "DB",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

missing = [
    c for c in required_cols
    if c not in train_df.columns
]

if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
    )


# ============================================================
# CREATE VESSEL-LEVEL GROUPS
# ============================================================

train_df["LAD"] = (
    train_df[
        ["PLAD", "MLAD", "DLAD", "DB"]
    ]
    .max(axis=1)
    .astype(int)
)

train_df["LCX"] = (
    train_df[
        ["PLCX", "MLCX", "DLCX", "OM"]
    ]
    .max(axis=1)
    .astype(int)
)

train_df["RCA"] = (
    train_df[
        ["PRCA", "MRCA", "DRCA"]
    ]
    .max(axis=1)
    .astype(int)
)


# ============================================================
# INITIAL SAMPLE WEIGHTS
# ============================================================

phenotype_weights = np.ones(
    len(train_df),
    dtype=np.float32
)


# ============================================================
# NSTEMI-OMI
# ============================================================

nstemi_omi_mask = (
    train_df["OMI"].eq(1)
    &
    train_df["NSTEMI"].eq(1)
)

phenotype_weights[
    nstemi_omi_mask.values
] = CONFIG[
    "nstemi_omi_weight"
]


# ============================================================
# HARD NSTEMI OMI:
# LCX OR RCA INVOLVEMENT
# ============================================================

hard_nstemi_omi_mask = (
    train_df["OMI"].eq(1)
    &
    train_df["NSTEMI"].eq(1)
    &
    (
        train_df["LCX"].eq(1)
        |
        train_df["RCA"].eq(1)
    )
)

phenotype_weights[
    hard_nstemi_omi_mask.values
] = CONFIG[
    "hard_nstemi_vessel_weight"
]


# ============================================================
# SUMMARY
# ============================================================

print("Phenotype weighting summary")
print("----------------------------------------")

print(
    "Weight 1.0:",
    int(
        np.sum(
            phenotype_weights == 1.0
        )
    )
)

print(
    f"Weight {CONFIG['nstemi_omi_weight']}:",
    int(
        np.sum(
            phenotype_weights
            == CONFIG[
                "nstemi_omi_weight"
            ]
        )
    )
)

print(
    f"Weight {CONFIG['hard_nstemi_vessel_weight']}:",
    int(
        np.sum(
            phenotype_weights
            == CONFIG[
                "hard_nstemi_vessel_weight"
            ]
        )
    )
)

print()

print(
    "NSTEMI-OMI total:",
    int(
        nstemi_omi_mask.sum()
    )
)

print(
    "Hard NSTEMI-LCX/RCA OMI:",
    int(
        hard_nstemi_omi_mask.sum()
    )
)

print()

print("Derived vessel counts:")
print(
    "LAD:",
    int(train_df["LAD"].sum())
)

print(
    "LCX:",
    int(train_df["LCX"].sum())
)

print(
    "RCA:",
    int(train_df["RCA"].sum())
)

Phenotype weighting summary
----------------------------------------
Weight 1.0: 14917
Weight 1.5: 131
Weight 2.0: 225

NSTEMI-OMI total: 356
Hard NSTEMI-LCX/RCA OMI: 225

Derived vessel counts:
LAD: 1091
LCX: 784
RCA: 1313


In [7]:
# CELL 6 — PHENOTYPE-AWARE DATASET

class PhenotypeOMIDataset(Dataset):
    def __init__(
        self,
        h5_path,
        csv_path,
        phenotype_weights=None
    ):
        self.h5_path = h5_path
        self.df = pd.read_csv(csv_path)

        with h5py.File(self.h5_path, "r") as f:
            ids = f["record_ids"][:]

            self.record_ids = [
                x.decode() if isinstance(x, bytes) else str(x)
                for x in ids
            ]

            self.length = len(f["signals"])

        assert self.length == len(self.df)

        csv_ids = self.df["record_id"].astype(str).str.zfill(5).tolist()
        h5_ids = [str(x).zfill(5) for x in self.record_ids]

        assert csv_ids == h5_ids, "CSV and H5 record order mismatch!"

        self.labels = self.df["OMI"].astype(np.float32).values

        if phenotype_weights is None:
            self.phenotype_weights = np.ones(
                self.length,
                dtype=np.float32
            )
        else:
            assert len(phenotype_weights) == self.length

            self.phenotype_weights = np.asarray(
                phenotype_weights,
                dtype=np.float32
            )

        self.h5 = None

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, "r")

        ecg = torch.tensor(
            self.h5["signals"][idx],
            dtype=torch.float32
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        phenotype_weight = torch.tensor(
            self.phenotype_weights[idx],
            dtype=torch.float32
        )

        return ecg, label, phenotype_weight


In [8]:
# CELL 7 — DATASETS + NORMAL DATALOADERS

train_dataset = PhenotypeOMIDataset(
    TRAIN_H5,
    TRAIN_CSV,
    phenotype_weights=phenotype_weights
)

val_dataset = PhenotypeOMIDataset(
    VAL_H5,
    VAL_CSV,
    phenotype_weights=None
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 955
Validation batches: 168


In [9]:
# CELL 8 — CLASS WEIGHT + TRAIN/VALIDATION CRITERIA

n_pos = np.sum(train_dataset.labels == 1)
n_neg = np.sum(train_dataset.labels == 0)

pos_weight_value = n_neg / n_pos

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE
)

criterion_train = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight,
    reduction="none"
)

criterion_val = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("non-OMI:", int(n_neg))
print("OMI:", int(n_pos))
print("Positive class weight:", round(float(pos_weight_value), 4))


non-OMI: 14288
OMI: 985
Positive class weight: 14.5056


In [10]:
# CELL 9 — CONFIGURE LAST2 FINE-TUNING

def configure_finetuning(model, mode="last2"):

    for param in model.parameters():
        param.requires_grad = False

    for param in model.dense.parameters():
        param.requires_grad = True

    if mode == "last2":

        for stage in model.stage_list[-2:]:
            for param in stage.parameters():
                param.requires_grad = True

    else:
        raise ValueError("Task 3 baseline must be 'last2'.")

    return model


model = configure_finetuning(
    model,
    CONFIG["baseline_mode"]
)

print("Fine-tuning mode:", CONFIG["baseline_mode"])

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)


Fine-tuning mode: last2
Trainable parameters: 26,709,793


In [11]:
# CELL 10 — OPTIMIZER

def build_optimizer(model, config):

    head_params = []
    backbone_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if name.startswith("dense"):
            head_params.append(param)
        else:
            backbone_params.append(param)

    return torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": config["backbone_lr"]
            },
            {
                "params": head_params,
                "lr": config["head_lr"]
            }
        ],
        weight_decay=config["weight_decay"]
    )


optimizer = build_optimizer(model, CONFIG)

model_param_ids = {
    id(p)
    for p in model.parameters()
    if p.requires_grad
}

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

assert model_param_ids == optimizer_param_ids, (
    "Optimizer is not connected to current model."
)

print("Optimizer correctly attached to current model.")

for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group["params"])

    print(
        f"Group {i}: "
        f"LR={group['lr']:.1e} | "
        f"Parameters={n_params:,}"
    )


Optimizer correctly attached to current model.
Group 0: LR=1.0e-05 | Parameters=26,708,768
Group 1: LR=1.0e-04 | Parameters=1,025


In [12]:
# CELL 11 — VALIDATION FUNCTION

def evaluate_model(
    model,
    loader,
    criterion,
    device,
    threshold=0.5
):

    model.eval()

    losses = []
    probabilities = []
    labels = []

    with torch.no_grad():

        for x, y, _ in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            logits = model(x).squeeze(1)

            loss = criterion(
                logits,
                y
            )

            probs = torch.sigmoid(
                logits
            )

            losses.append(
                loss.item()
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                y.cpu().numpy()
            )

    probabilities = np.asarray(
        probabilities
    )

    labels = np.asarray(
        labels
    ).astype(int)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    auroc = roc_auc_score(
        labels,
        probabilities
    )

    auprc = average_precision_score(
        labels,
        probabilities
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0
    )

    return {
        "loss": float(np.mean(losses)),
        "auroc": float(auroc),
        "auprc": float(auprc),
        "f1": float(f1),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "probabilities": probabilities,
        "labels": labels
    }


In [13]:
# CELL 12 — TASK 3 TRAINING FUNCTION

def train_model_task3(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion_train,
    criterion_val,
    device,
    config
):

    best_auprc = -1.0
    epochs_without_improvement = 0
    history = []

    save_path = os.path.join(
        config["save_dir"],
        "task3_ecgfounder_phenotype_weighting_last2.pth"
    )

    for epoch in range(
        1,
        config["epochs"] + 1
    ):

        # ==========================
        # TRAIN
        # ==========================

        model.train()

        train_losses = []

        train_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch:02d}/{config['epochs']} - Training",
            leave=False
        )

        for x, y, phenotype_weight in train_bar:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            phenotype_weight = phenotype_weight.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(x).squeeze(1)

            individual_losses = criterion_train(
                logits,
                y
            )

            loss = (
                individual_losses
                * phenotype_weight
            ).mean()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config["max_grad_norm"]
            )

            optimizer.step()

            train_losses.append(
                loss.item()
            )

            train_bar.set_postfix(
                current_loss=f"{loss.item():.4f}",
                avg_loss=f"{np.mean(train_losses):.4f}"
            )

        train_loss = float(
            np.mean(train_losses)
        )

        # ==========================
        # VALIDATION
        # ==========================

        metrics = evaluate_model(
            model,
            val_loader,
            criterion_val,
            device
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": metrics["loss"],
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
                "f1": metrics["f1"],
                "sensitivity": metrics["sensitivity"],
                "specificity": metrics["specificity"]
            }
        )

        print("\n" + "=" * 65)

        print(
            "TASK 3 — PHENOTYPE-AWARE WEIGHTING"
        )

        print(
            "Baseline mode:",
            config["baseline_mode"]
        )

        print(
            f"Epoch {epoch:02d}/"
            f"{config['epochs']}"
        )

        print(
            f"Train Loss : "
            f"{train_loss:.5f}"
        )

        print(
            f"Val Loss   : "
            f"{metrics['loss']:.5f}"
        )

        print(
            f"AUROC      : "
            f"{metrics['auroc']:.4f}"
        )

        print(
            f"AUPRC      : "
            f"{metrics['auprc']:.4f}"
        )

        print(
            f"F1         : "
            f"{metrics['f1']:.4f}"
        )

        print(
            f"Sensitivity: "
            f"{metrics['sensitivity']:.4f}"
        )

        print(
            f"Specificity: "
            f"{metrics['specificity']:.4f}"
        )

        print(
            f"TN={metrics['tn']} "
            f"FP={metrics['fp']} "
            f"FN={metrics['fn']} "
            f"TP={metrics['tp']}"
        )

        # ==========================
        # SAVE BEST BY AUPRC
        # ==========================

        if metrics["auprc"] > best_auprc:

            best_auprc = metrics["auprc"]
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "epoch":
                        epoch,

                    "task":
                        "phenotype_weighting",

                    "baseline_mode":
                        config["baseline_mode"],

                    "val_auprc":
                        metrics["auprc"],

                    "val_auroc":
                        metrics["auroc"],

                    "nstemi_omi_weight":
                        config["nstemi_omi_weight"],

                    "hard_nstemi_vessel_weight":
                        config["hard_nstemi_vessel_weight"],

                    "config":
                        config
                },

                save_path
            )

            print(
                "\n✓ New best Task-3 model saved"
            )

            print(
                "Path:",
                save_path
            )

        else:

            epochs_without_improvement += 1

            print(
                f"\nNo improvement: "
                f"{epochs_without_improvement}/"
                f"{config['patience']}"
            )

            if (
                epochs_without_improvement
                >= config["patience"]
            ):

                print(
                    "\nEarly stopping triggered."
                )

                break

        print("=" * 65)

    print("\nTraining finished.")

    print(
        "Best validation AUPRC:",
        round(best_auprc, 4)
    )

    print(
        "Best checkpoint:",
        save_path
    )

    return history


In [14]:
# CELL 13 — START TASK 3

history = train_model_task3(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion_train=criterion_train,
    criterion_val=criterion_val,
    device=DEVICE,
    config=CONFIG
)


Epoch 01/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 3 — PHENOTYPE-AWARE WEIGHTING
Baseline mode: last2
Epoch 01/30
Train Loss : 1.18803
Val Loss   : 0.87384
AUROC      : 0.8571
AUPRC      : 0.3204
F1         : 0.3185
Sensitivity: 0.7349
Specificity: 0.8102
TN=2041 FP=478 FN=44 TP=122

✓ New best Task-3 model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task3_phenotype_weighting/task3_ecgfounder_phenotype_weighting_last2.pth


Epoch 02/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 3 — PHENOTYPE-AWARE WEIGHTING
Baseline mode: last2
Epoch 02/30
Train Loss : 1.02060
Val Loss   : 0.89469
AUROC      : 0.8665
AUPRC      : 0.3455
F1         : 0.3540
Sensitivity: 0.7048
Specificity: 0.8499
TN=2141 FP=378 FN=49 TP=117

✓ New best Task-3 model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task3_phenotype_weighting/task3_ecgfounder_phenotype_weighting_last2.pth


Epoch 03/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]

...


TASK 3 — PHENOTYPE-AWARE WEIGHTING
Baseline mode: last2
Epoch 11/30
Train Loss : 0.62157
Val Loss   : 1.18141
AUROC      : 0.8677
AUPRC      : 0.3519
F1         : 0.3811
Sensitivity: 0.5843
Specificity: 0.9023
TN=2273 FP=246 FN=69 TP=97

No improvement: 3/5


Epoch 12/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 3 — PHENOTYPE-AWARE WEIGHTING
Baseline mode: last2
Epoch 12/30
Train Loss : 0.56428
Val Loss   : 1.27739
AUROC      : 0.8636
AUPRC      : 0.3422
F1         : 0.3934
Sensitivity: 0.5783
Specificity: 0.9103
TN=2293 FP=226 FN=70 TP=96

No improvement: 4/5


Epoch 13/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 3 — PHENOTYPE-AWARE WEIGHTING
Baseline mode: last2
Epoch 13/30
Train Loss : 0.48851
Val Loss   : 1.38292
AUROC      : 0.8580
AUPRC      : 0.3317
F1         : 0.3958
Sensitivity: 0.5663
Specificity: 0.9146
TN=2304 FP=215 FN=72 TP=94

No improvement: 5/5

Early stopping triggered.

Training finished.
Best validation AUPRC: 0.3748
Best checkpoint: /content/drive/MyDrive/ECG_Project/OMI_model/task3_phenotype_weighting/task3_ecgfounder_phenotype_weighting_last2.pth


## Clean ablation rule

For Task 3, keep unchanged:
- ECG preprocessing
- patient-safe train/validation split
- ECGFounder architecture
- Last2 fine-tuning strategy
- learning rates
- AdamW optimizer
- original positive-class weighting
- batch size
- early stopping
- checkpoint selection by validation AUPRC

The only intended experimental change is phenotype-aware loss weighting.
